In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet
import os
import numpy as np
import json
from pycocotools.coco import COCO
import cv2
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

DATA_ROOT = r"C:/RemoteStorage/UoM/Year 3/ARI3129 Advanced CV for AI/Object Detection Models/Datasets/COCO-based_COCO_mounting" 

MODEL_NAME = 'tf_efficientdet_d0'
NUM_CLASSES = 2
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
class CocoEffDetDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, set_name='train', img_size=512, transforms=None):
        self.root_dir = root_dir
        self.set_name = set_name
        
        self.img_dir = os.path.join(root_dir, 'images', set_name)
        
        self.ann_file = os.path.join(root_dir, 'annotations', f'{set_name}.json')
        
        self.coco = COCO(self.ann_file)
        self.img_ids = list(sorted(self.coco.imgs.keys()))
        
        self.img_size = img_size
        self.transforms = transforms

        cats = self.coco.loadCats(self.coco.getCatIds())
        self.cat_id_to_label = {cat['id']: i + 1 for i, cat in enumerate(cats)}

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        
        file_name = os.path.basename(img_info['file_name'])
        img_path = os.path.join(self.img_dir, file_name)
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        coco_anns = self.coco.loadAnns(ann_ids)
        
        bboxes = []
        labels = []
        
        for ann in coco_anns:
            x_min, y_min, w, h = ann['bbox']
            bboxes.append([x_min, y_min, x_min + w, y_min + h])
            labels.append(self.cat_id_to_label[ann['category_id']])

        if self.transforms:
            if len(bboxes) > 0:
                transformed = self.transforms(image=image, bboxes=bboxes, labels=labels)
                image = transformed['image']
                bboxes = transformed['bboxes']
                labels = transformed['labels']
            else:
                transformed = self.transforms(image=image, bboxes=[], labels=[])
                image = transformed['image']
                bboxes = []
                labels = []

        boxes_tensor = torch.as_tensor(bboxes, dtype=torch.float32)
        labels_tensor = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {}
        target['image_id'] = torch.tensor([img_id])
        if len(boxes_tensor) > 0:
             target['bbox'] = boxes_tensor[:, [1, 0, 3, 2]] 
        else:
             target['bbox'] = torch.zeros((0, 4), dtype=torch.float32)
             
        target['cls'] = labels_tensor
        target["img_scale"] = torch.tensor([1.0])
        target["img_size"] = torch.tensor([img_info['height'], img_info['width'], 3])

        return image, target

    def __len__(self):
        return len(self.img_ids)

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)

    max_boxes = max([t['bbox'].shape[0] for t in targets])
    max_boxes = max(max_boxes, 1)

    batch_bboxes = []
    batch_classes = []
    batch_scales = []
    batch_sizes = []

    for t in targets:
        boxes = t['bbox']
        classes = t['cls']
        num_objs = boxes.shape[0]
        
        pad_len = max_boxes - num_objs
        
        if pad_len > 0:
            boxes_pad = torch.zeros((pad_len, 4), dtype=torch.float32)
            boxes = torch.cat([boxes, boxes_pad], dim=0)
            
            classes_pad = torch.ones((pad_len,), dtype=torch.int64) * -1
            classes = torch.cat([classes, classes_pad], dim=0)
        
        batch_bboxes.append(boxes)
        batch_classes.append(classes)
        batch_scales.append(t['img_scale'])
        batch_sizes.append(t['img_size'])

    target_dict = {
        'bbox': torch.stack(batch_bboxes),
        'cls': torch.stack(batch_classes),
        'img_scale': torch.stack(batch_scales),
        'img_size': torch.stack(batch_sizes),
        'image_id': torch.cat([t['image_id'] for t in targets])
    }
    
    return images, target_dict

In [ ]:
train_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(p=1.0),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

val_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(p=1.0),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

train_ds = CocoEffDetDataset(DATA_ROOT, 'train', img_size=IMG_SIZE, transforms=train_transform)
val_ds = CocoEffDetDataset(DATA_ROOT, 'val', img_size=IMG_SIZE, transforms=val_transform)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
DataLoaders updated with padding logic.
Training samples: 513
Validation samples: 100


In [ ]:
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet
from omegaconf import OmegaConf

config = get_efficientdet_config(MODEL_NAME)

OmegaConf.set_readonly(config, False)

# 2. Modify Config
config.image_size = (IMG_SIZE, IMG_SIZE)
config.norm_kwargs = dict(eps=.001, momentum=.01)
config.num_classes = NUM_CLASSES 

net = EfficientDet(config, pretrained_backbone=True)

net.class_net = HeadNet(config, num_outputs=NUM_CLASSES)

bench = DetBenchTrain(net, config)
bench = bench.to(DEVICE)

optimizer = torch.optim.AdamW(bench.parameters(), lr=LEARNING_RATE, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, verbose=True)

print(f"Model {MODEL_NAME} initialized for {NUM_CLASSES} classes.")

INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/tf_efficientnet_b0.ns_jft_in1k)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/timm/tf_efficientnet_b0.ns_jft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO:timm.models._hub:[timm/tf_efficientnet_b0.ns_jft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


Model tf_efficientdet_d0 initialized for 2 classes.


In [17]:
model_save_path = "best_mounting_model.pth"
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    bench.train()
    train_losses = []
    
    for images, targets in train_loader:
        images = images.to(DEVICE)
        targets = {k: v.to(DEVICE) for k, v in targets.items()}

        optimizer.zero_grad()
        loss_dict = bench(images, targets)
        loss = loss_dict['loss']
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())

    avg_train_loss = np.mean(train_losses)
    val_losses = []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(DEVICE)
            targets = {k: v.to(DEVICE) for k, v in targets.items()}
            
            loss_dict = bench(images, targets)
            val_losses.append(loss_dict['loss'].item())

    avg_val_loss = np.mean(val_losses)
    
    print(f"Epoch {epoch+1}: {avg_train_loss:.4f} | {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(bench.model.state_dict(), model_save_path)

print(f"Best Val Loss: {best_val_loss:.4f}")

Epoch 1: 88.1749 | 2.2192
Epoch 2: 1.4908 | 1.1274
Epoch 3: 0.8887 | 0.8296
Epoch 4: 0.6296 | 0.6116
Epoch 5: 0.5040 | 0.5242
Epoch 6: 0.4216 | 0.4910
Epoch 7: 0.3891 | 0.4247
Epoch 8: 0.3343 | 0.3849
Epoch 9: 0.2961 | 0.3844
Epoch 10: 0.2907 | 0.3767
Epoch 11: 0.2656 | 0.3338
Epoch 12: 0.2569 | 0.3568
Epoch 13: 0.2410 | 0.3123
Epoch 14: 0.2227 | 0.3184
Epoch 15: 0.2206 | 0.2900
Epoch 16: 0.2128 | 0.2858
Epoch 17: 0.2008 | 0.2785
Epoch 18: 0.1904 | 0.3241
Epoch 19: 0.1973 | 0.2594
Epoch 20: 0.1638 | 0.2870
Best Val Loss: 0.2594


In [ ]:
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet
import os
import numpy as np
import json
from pycocotools.coco import COCO
import cv2
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

DATA_ROOT = r"C:/RemoteStorage/UoM/Year 3/ARI3129 Advanced CV for AI/Object Detection Models/Datasets/COCO-based_COCO_mounting" 

MODEL_NAME = 'tf_efficientdet_d0'
NUM_CLASSES = 2
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

DATA_ROOT = r"C:/RemoteStorage/UoM/Year 3/ARI3129 Advanced CV for AI/Object Detection Models/Datasets/COCO-based_COCO_mounting" 

MODEL_NAME = 'tf_efficientdet_d0'
NUM_CLASSES = 2
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
from pathlib import Path

WEIGHTS_PATH = 'best_mounting_model.pth' 
ROOT_DIR = Path(r'C:\RemoteStorage\UoM\Year 3\ARI3129 Advanced CV for AI\Object Detection Models')
DATA_DIR = ROOT_DIR / 'Datasets' / 'COCO-based_COCO_mounting'
TEST_IMG_DIR = DATA_DIR / 'images' / 'test'
TEST_ANN_FILE = DATA_DIR / 'annotations' / 'test.json'

IMG_SIZE = 512       
NUM_CLASSES = 2
BATCH_SIZE = 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

class CocoJsonDataset(Dataset):
    def __init__(self, img_dir, json_file):
        self.img_dir = img_dir
        
        with open(json_file, 'r') as f:
            self.coco_data = json.load(f)
            
        self.img_id_to_anns = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_id_to_anns:
                self.img_id_to_anns[img_id] = []
            self.img_id_to_anns[img_id].append(ann)
            
        self.images = self.coco_data['images']

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        
        img_path = os.path.join(self.img_dir, file_name)
        img = cv2.imread(img_path)
        
        if img is None:
            return torch.zeros((3, IMG_SIZE, IMG_SIZE)), {'bbox': torch.zeros((0,4)), 'cls': torch.zeros((0,))}

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_orig, w_orig, _ = img.shape
        
        anns = self.img_id_to_anns.get(img_id, [])
        
        boxes = []
        labels = []
        
        for ann in anns:
            x, y, w, h = ann['bbox']
            
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id']) 

        target = {}
        if len(boxes) > 0:
            target['bbox'] = torch.as_tensor(boxes, dtype=torch.float32)
            target['cls'] = torch.as_tensor(labels, dtype=torch.float32)
        else:
            target['bbox'] = torch.zeros((0, 4), dtype=torch.float32)
            target['cls'] = torch.zeros((0,), dtype=torch.float32)
            
        img_t = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img_t = torch.from_numpy(img_t).permute(2, 0, 1).float() / 255.0
        
        scale_x = IMG_SIZE / w_orig
        scale_y = IMG_SIZE / h_orig
        if len(boxes) > 0:
            target['bbox'][:, [0, 2]] *= scale_x
            target['bbox'][:, [1, 3]] *= scale_y
            
        return img_t, target

test_ds = CocoJsonDataset(TEST_IMG_DIR, TEST_ANN_FILE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)), num_workers=0)

config = get_efficientdet_config('tf_efficientdet_d0')
net = EfficientDet(config, pretrained_backbone=False)

net.reset_head(num_classes=2) 
net.class_net = HeadNet(config, num_outputs=config.num_classes)

state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE)
net.load_state_dict(state_dict)

bench_predict = DetBenchPredict(net).to(DEVICE)
bench_predict.eval()
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=False)

with torch.no_grad():
    for images, targets in test_loader:
        images = torch.stack(images).to(DEVICE)
        output = bench_predict(images)
        
        preds = []
        for i in range(len(output)):
            keep = output[i, :, 4] > 0.05
            
            preds.append(dict(
                boxes=output[i, keep, :4],
                scores=output[i, keep, 4],
                labels=output[i, keep, 5].int() - 1 
            ))
            
        target_list = [
            dict(boxes=t['bbox'].to(DEVICE), labels=(t['cls'] - 1).int().to(DEVICE)) 
            for t in targets
        ]
        metric.update(preds, target_list)

results = metric.compute()

map_50 = results['map_50'].item()
map_50_95 = results['map'].item()
recall_mean = results['mar_100'].item()

print(f"mAP @ 50% IoU:           {map_50:.4f}")
print(f"mAP @ 50-95% IoU:        {map_50_95:.4f}")
print(f"Precision (Approx):      {map_50:.4f}") 
print(f"Recall (Mean):           {recall_mean:.4f}")

Parsing annotations from C:\RemoteStorage\UoM\Year 3\ARI3129 Advanced CV for AI\Object Detection Models\Datasets\COCO-based_COCO_mounting\annotations\test.json...
Loading EfficientDet-D0 from best_mounting_model.pth...
✅ Weights loaded successfully.
Running evaluation on test set...


C:\Users\Roman\AppData\Local\Temp\ipykernel_13908\3300242940.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE)


FINAL TEST SET RESULTS (EfficientDet - Mounting)
mAP @ 50% IoU:           0.6326
mAP @ 50-95% IoU:        0.4025
----------------------------------------
Precision (Approx):      0.6326
Recall (Mean):           0.5651
